# Cyclistic Bike-Share — Transparent secondary-aggregate analysis

**Attribution:** Figures here are recalculated from publicly displayed summary tables in Daniel Eniola’s [Cyclistic analysis](https://rstudio-pubs-static.s3.amazonaws.com/1411490_e71e5ba643c44e6ea9333d53b9b1b9e6.html). The source claims inconsistent dates; the 12 month labels are therefore shown without certifying the year. **This notebook does not claim to have independently processed all Divvy trip-level rows.** Original provider: [Divvy system data](https://divvybikes-marketing.lyft.net/system-data); [license](https://divvybikes.com/data-license-agreement).

## 1 — Ask
How do casual riders differ from annual members in trip volume, ride length, weekly and seasonal distribution? Which pilot marketing campaigns would be reasonable to test?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

MONTHS = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
WEEKDAYS = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

## 2 — Prepare
The following aggregate tables are transcribed from a *credited third-party published report* and validated against its displayed totals.

In [ ]:
monthly = pd.DataFrame({'month': ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'], 'casual': [23964, 27051, 83064, 105528, 176162, 279570, 309293, 324165, 255223, 214849, 94930, 27173], 'member': [109969, 122118, 208500, 257962, 314091, 379609, 430535, 443214, 441046, 414231, 252033, 109411]})
weekly = pd.DataFrame({'weekday': ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'], 'casual': [220280, 218800, 212183, 247670, 307148, 396411, 318480], 'member': [494937, 558517, 538722, 560394, 517005, 438652, 374492], 'casual_avg_min': [23.2, 20.6, 19.4, 20.7, 23.5, 26.4, 27.3], 'member_avg_min': [12.2, 12.2, 12.1, 12.2, 12.7, 13.9, 13.9]})
monthly.head()

## 3 — Process
Check within-category sums agree in both sources. Calculate proportions. This is processing of summary tables, not raw-trip cleaning.

In [ ]:
totals = monthly[["member", "casual"]].sum()
assert totals.equals(weekly[["member", "casual"]].sum())
assert totals.sum() == 5_403_691
print("Counts by type:", totals.to_dict())
print("Shares:", (totals/totals.sum()*100).round(2).to_dict())
print("Weekend shares:", (weekly.loc[weekly.weekday.isin(["Sat", "Sun"]), ["member", "casual"]].sum()/totals*100).round(2).to_dict())

## 4 — Analyze / Share
Show trip mix, monthly patterns and weekday volume. In an actual submission, verify these findings against the trip-level CSVs; analyze median trip duration and daily hours too.

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(12,4))
monthly.set_index("month")[["member","casual"]].plot(ax=axs[0],marker="o",title="Monthly trips")
weekly.set_index("weekday")[["member","casual"]].plot(ax=axs[1],kind="bar",title="Trips by weekday")
for ax in axs: ax.set_ylabel("Number of trips"); ax.tick_params(axis="x",rotation=30)
plt.tight_layout();plt.show()

## 5 — Act (pilots, not proven effects)
1. A/B-test weekend membership messaging.
2. Test a pre-summer seasonal offer with a control group.
3. Test weekday convenience and price break-even messages.

**Success metrics:** attributable incremental annual sign-ups, CAC, 30/90-day retention, incremental contribution after discounts. This trip table cannot identify unique riders or prove why a trip was taken.

## Optional: Independently analyze raw monthly files once you download them from the official Divvy interface
The cells below are NOT executed in this notebook, because the original CSVs were not accessible here. Put 12 files under `raw/`, use a consistent date window, and run the following pipeline. Record all cleaning decisions and row counts.

In [ ]:
# OPTIONAL — requires raw/*.csv downloaded separately
# from pathlib import Path
# files=sorted(Path("raw").glob("*.csv"))
# assert len(files)==12, "Supply the 12 selected monthly CSV files"
# rides=pd.concat((pd.read_csv(f, low_memory=False) for f in files),ignore_index=True)
# for col in ["started_at", "ended_at"]:
#     rides[col]=pd.to_datetime(rides[col],errors="coerce")
# print("Input",len(rides),"duplicated IDs",rides.ride_id.duplicated().sum())
# rides["duration_min"]=(rides.ended_at-rides.started_at).dt.total_seconds()/60
# bad=rides["duration_min"].isna() | (rides["duration_min"]<1)
# cleaned=rides.loc[~bad & rides.member_casual.isin(["member","casual"])].copy()
# print("Excluded invalid duration:",int(bad.sum()),"Cleaned:",len(cleaned))
# cleaned["weekday"]=cleaned.started_at.dt.day_name()
# cleaned["month"]=cleaned.started_at.dt.to_period("M").astype(str)
# print(cleaned.groupby("member_casual").duration_min.agg(["count","mean","median"]))
# print(cleaned.groupby(["member_casual","weekday"]).size())